# 00 - הקמת סביבה ואיסוף נתונים

**פרויקט: תחנות קריטיות - מרכזיות (centrality) וחוסן ברשת התחבורה הציבורית בישראל**

התחבורה הציבורית הארצית בישראל מתפרסמת על ידי משרד התחבורה כפיד **GTFS**
(General Transit Feed Specification): חבילה של קבצי CSV טקסטואליים המתארים כל
מפעיל, קו, נסיעה, תחנה וזמן עצירה מתוזמן במדינה. הפיד הזה הוא למעשה גרף בתחפושת.
אם נתייחס לכל **תחנה** כאל צומת ונחבר שתי תחנות בכל פעם שנסיעה מתוזמנת כלשהי עוברת
ישירות מהאחת לשנייה, נקבל רשת ניידות ארצית של כ-30,000 צמתים ו-52,000 קשתות.

סדרת המחברות הזו שואלת שתי שאלות על הרשת הזו:

1. **אילו תחנות הן קריטיות?** אילו תחנות נושאות חלק לא פרופורציונלי מהקישוריות של
   המדינה, על פי degree, degree משוקלל, PageRank, betweenness, נקודות חיתוך
   (articulation points) וגשרים (bridges)?
2. **עד כמה הרשת חסינה לאובדנן?** אם התחנות המדורגות בראש מוסרות - הצפה, אירוע
   ביטחוני, שביתה, עבודות בנייה - באיזו מהירות הרשת מתפרקת בהשוואה לאובדן תחנות
   אקראיות?

המחברת הראשונה הזו היא **נקודת הכניסה**. היא אינה מבצעת תורת גרפים כלל. תפקידה
להוכיח שהסביבה עובדת, להביא אל הדיסק את קובץ הנתונים הגדול היחיד שאינו מנוהל
ב-git, ולתת לקורא מצאי כן של הנתונים הגולמיים שעליהם נבנה כל השאר.

**קלט**

- `israel-public-transportation/` - פיד ה-GTFS המסופק יחד עם המאגר
  (`agency.txt`, `calendar.txt`, `fare_attributes.txt`, `fare_rules.txt`, `routes.txt`,
  `stops.txt`, `translations.txt`, `trips.txt`)
- `stop_times.txt` (816 MB) - **אינו** ב-git; מורד לפי דרישה מ-Google Drive

**פלט** (הכול תחת `outputs/nb/00_setup_and_data/`)

- `tables/gtfs_file_inventory.csv` - כל קובץ בפיד עם גודל, מספר שורות ותפקיד
- `tables/route_type_distribution.csv` - מספר קווים לפי `route_type` של GTFS
- `tables/agency_route_counts.csv` - מספר קווים לפי מפעיל
- `tables/service_calendar_summary.csv` - ימי שירות לפי יום בשבוע
- `feed_manifest.json` - סיכום קריא-מכונה שנצרך על ידי המחברות הבאות
- `figures/route_type_distribution.png`, `figures/routes_per_agency.png`,
  `figures/stop_locations.png`

**תלוי ב:** בכלום. זהו שלב 00 - יש להריץ אותו ראשון.

**זמן ריצה:** מספר שניות, בתוספת הורדה חד-פעמית של כ-816 MB אם `stop_times.txt` חסר.

## 1. אתחול הסביבה

התא שלהלן הוא קטע ה-boilerplate היחיד בפרויקט, והוא חוזר מילה במילה בראש כל מחברת.
הוא מבצע שלושה דברים:

- `_ensure(...)` מתקין חבילה **רק אם ה-import אינו ניתן לפתרון**, כך שהרצה חוזרת של
  מחברת על מכונה שכבר מוגדרת אינה עולה דבר ואינה פונה לרשת כלל.
- `find_repo_root()` מטפס כלפי מעלה מהתיקייה הנוכחית בחיפוש אחר התיקייה
  `israel-public-transportation/`. הדבר מאפשר למחברת לעבוד בין אם מפעילים את Jupyter
  משורש המאגר, מתוך `notebooks/`, או מכל מקום אחר. אם התיקייה אינה נמצאת - וזה מה
  שקורה בסביבת ריצה טרייה של Google Colab - הקוד משכפל (clone) את המאגר הציבורי
  אל `/content` במקום זאת. לפיכך הבודק אינו זקוק לדבר מלבד מחברת זו וחיבור לאינטרנט.
- הוא מקבע את תיקיית העבודה ומגדיר את `DATA` (הפיד הגולמי) ואת `OUT` (שורש הפלט של
  המחברת), כך שכל תא בהמשך יכול להשתמש באובייקטי `Path` ללא נתיבים יחסיים.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## 2. תלויות ובדיקת גרסאות

הפרויקט כולו רץ על מחסנית scientific-Python קטנה ורגילה: **pandas** לטבלאות ה-CSV,
**numpy** לחישובים המספריים, **matplotlib** לכל האיורים, ו-**networkx** לעבודת הגרפים
במחברות הבאות. אנו מתקינים אותן כאן (ללא פעולה אם הן כבר קיימות) ואז מדפיסים את
הגרסאות.

הדפסת הגרסאות אינה קישוט. זו הדרך המהירה ביותר עבור בודק לראות שהסביבה אכן פעילה,
ואם תוצאה כלשהי לא תשוחזר בעתיד, שורת הגרסאות היא הדבר הראשון שכדאי להשוות.



In [ ]:
_ensure("pandas", "numpy", "matplotlib", "networkx")

import json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import networkx as nx

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

print("python     ", sys.version.split()[0])
print("pandas     ", pd.__version__)
print("numpy      ", np.__version__)
print("matplotlib ", matplotlib.__version__)
print("networkx   ", nx.__version__)
print("\nData dir exists:", DATA.is_dir(), "->", DATA)

## 3. מוסכמות פלט ומתגי זמן ריצה

כל מחברת בסדרה כותבת לתיקיית השלב **שלה** תחת `outputs/nb/`, וקוראת את השלבים
הקודמים מתיקיותיהם. דבר אינו נכתב אל `outputs/tables`, `outputs/figures` או
`outputs/rail` - אלה מכילים את התוצאות המצוטטות כבר בדוח הכתוב וחייבים להישאר ללא
שינוי.

שני קבועים שולטים בעלות הלא-טריוויאלית היחידה במחברת זו:

- `COUNT_ROWS` - ספירת השורות של `stop_times.txt` פירושה קריאה זורמת של 816 MB
  מהדיסק (בערך 5-30 שניות, בהתאם לכונן). יש להגדיר אותו כ-`False` להרצה מיידית;
  במקרה כזה המצאי ידווח את מספר השורות כ-`NaN` עבור אותו קובץ בלבד.
- `PREVIEW_ROWS` / `TOP_AGENCIES` - מגבלות תצוגה קוסמטיות בלבד.

In [ ]:
STAGE = OUT / "00_setup_and_data"
(STAGE / "tables").mkdir(parents=True, exist_ok=True)
(STAGE / "figures").mkdir(parents=True, exist_ok=True)

# --- run-time switches -------------------------------------------------------
COUNT_ROWS = True      # False -> skip the ~816 MB scan of stop_times.txt
PREVIEW_ROWS = 5       # rows shown in each preview table
TOP_AGENCIES = 12      # bars in the "routes per operator" figure

print("Stage output folder:", STAGE)

## 4. הבאת הפיד הגולמי (`stop_times.txt`)

שמונה מתוך תשעת קבצי ה-GTFS קטנים דיים כדי לשכון במאגר. התשיעי, `stop_times.txt`,
שוקל **816 MB** - שורה אחת לכל הגעה מתוזמנת לכל תחנה במדינה, כ-15.7 מיליון שורות -
ו-git אינו המקום הנכון עבורו. הקובץ מאוחסן ב-Google Drive ונמשך לפי דרישה בעזרת
`gdown`.

קובץ זה הוא חומר הגלם של הגרף: שורות עוקבות החולקות אותו `trip_id` הן בדיוק אירועי
"כלי הרכב נסע ישירות מתחנה A לתחנה B" שהופכים לקשתות במחברת 02. ההורדה מתבצעת פעם
אחת; בכל הרצה עתידית בדיקת `exists()` עוקפת אותה.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## 5. טקסט בעברית באיורים

כל שם תחנה, תיאור קו ושם מפעיל בפיד הזה הם בעברית, והעברית נכתבת מימין לשמאל.
Matplotlib מרנדר מחרוזת בסדר לוגי (סדר האחסון) ואינו מיישם את האלגוריתם הדו-כיווני
(bidirectional) של Unicode, ולכן תווית בעברית שמצוירת בתמימות יוצאת עם אותיות
בסדר הפוך ובלתי קריאה לכל מי שיודע לקרוא עברית באמת.

הפתרון הוא לסדר מחדש את התווים לסדר *תצוגה* לפני שהם מגיעים ל-renderer, באמצעות
`python-bidi`. במקום לזכור לעטוף כל תווית ידנית, אנו מבצעים monkey-patch אחד ל-
`matplotlib.text.Text.set_text` כך שכל מחרוזת עברית מומרת אוטומטית, בכל מקום -
תוויות צירים, כותרות והערות. ה-patch הוא אידמפוטנטי (מוגן על ידי `_bidi_patched`)
ומשאיר טקסט שאינו עברי ללא כל שינוי.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## 6. מהו GTFS, וכיצד החלקים מתחברים זה לזה

GTFS הוא סכימה רלציונית המאוחסנת כקבצי CSV. הבנת הצירופים (joins) היא כל הסוד,
משום שהגרף שמעניין אותנו מתגלה רק לאחר שניים מהם:

```
agency.txt   1 --- *  routes.txt   1 --- *  trips.txt   1 --- *  stop_times.txt  * --- 1  stops.txt
 (operator)             (a line)              (one run of         (one arrival at        (a physical
                                               that line on        one stop, with         stop / platform)
                                               a given service)    a sequence number)
calendar.txt 1 --- * trips.txt        (which days a service_id actually runs)
fare_rules / fare_attributes          (ticketing - price zones, not topology)
translations.txt                      (Hebrew <-> other-language name strings)
```

יש לקרוא מלמטה למעלה: **תחנה** (stop) היא מיקום פיזי עם קואורדינטות. **stop_time**
אומר "נסיעה T הגיעה לתחנה S כעצירתה ה-k, בשעה 07:14". **נסיעה** (trip) היא הרצה
פיזית אחת של **קו** (route) ביום **שירות** (calendar) נתון, וקו שייך ל**מפעיל**
(agency).

כלל בניית הקשתות המשמש לאורך הפרויקט נובע ישירות מהסכימה הזו:

> יש למיין את `stop_times.txt` לפי `trip_id`, ולאחר מכן לפי `stop_sequence`. כל שתי
> **שורות עוקבות של אותה נסיעה** מתארות כלי רכב הנוסע ישירות מתחנה אחת לבאה אחריה -
> וזוהי קשת מכוונת. מספר הנסיעות המשתמשות באותו זוג תחנות מסודר הופך למשקל הקשת
> (`frequency`).

מכיוון שהפיד המפורסם מגיע כבר ממוין בצורה זו, ניתן לקרוא את הקובץ שורה אחר שורה
בזרימה ואין צורך להחזיקו בזיכרון - דבר החשוב מאוד כאשר מדובר ב-15.7 מיליון שורות.

**אילו קבצים הפרויקט הזה משתמש בהם בפועל:**

| קובץ | בשימוש? | תפקיד בפרויקט |
|---|---|---|
| `stop_times.txt` | **כן, ליבה** | מקור כל הקשתות: תחנות עוקבות בתוך נסיעה |
| `stops.txt` | **כן, ליבה** | תכונות הצומת: `stop_id`, שם בעברית, lat/lon, אזור |
| `trips.txt` | **כן** | ממפה נסיעות לקווים ולשירותים; מאפשר פילוח לפי אמצעי תחבורה |
| `routes.txt` | **כן** | `route_type` (אוטובוס / רכבת / רכבת קלה) - מניע את הניתוח לרכבות בלבד |
| `agency.txt` | כן, הקשר | שמות מפעילים לצורך פילוחים תיאוריים |
| `calendar.txt` | כן, הקשר | מאשש איזה חלון שירות מכוסה בתצלום המצב |
| `translations.txt` | לא | מחרוזות שמות רב-לשוניות; אנו עובדים בעברית ישירות |
| `fare_rules.txt` | לא | אזורי כרטוס - יחס תמחורי, לא טופולוגי |
| `fare_attributes.txt` | לא | מחירי כרטיסים - מאותה סיבה |

תעריפי הנסיעה הוצאו מהיקף העבודה במכוון. אזור תעריף מלמד כמה עולה נסיעה, ולא האם
הנסיעה אפשרית פיזית, והפרויקט הזה עוסק בקישוריות ובכשל.

## 7. מצאי של כל תשעת קבצי הפיד

כעת אנו מוכיחים שהנתונים אכן קיימים. עבור כל אחד מתשעת הקבצים אנו רושמים את גודלו
בדיסק, את רשימת העמודות שלו ואת מספר שורות הנתונים שבו.

ספירת השורות מחייבת מעט זהירות. `pd.read_csv` על קובץ בגודל 816 MB אמנם יעבוד, אך
הוא בזבזני כאשר כל שאנו רוצים הוא ספירה; לכן במקום זאת אנו קוראים את הקובץ בזרימה
במקטעים בינאריים של 8 MB וסופרים בתי newline, תוך החסרת אחד עבור שורת הכותרת
(ותוך טיפול במקרה של newline חסר בסוף הקובץ). הסתייגות אחת, שתיאמר בכנות: זו ספירת
*שורות*, לא רשומות CSV, ולכן שדה המכיל newline משובץ בתוך מרכאות היה נספר ביתר.
בדיקות מדגמיות מול הטבלאות המפורסרות מראות שבפיד זה אין מקרים כאלה.

עמודת `purpose` היא הערה משלנו - שיפוט ה"בשימוש / לא בשימוש" מסעיף הסכימה לעיל,
מצורף לנתונים כדי שהשלבים הבאים והדוח יוכלו לצטט אותו.

In [ ]:
FILE_PURPOSE = {
    "stop_times.txt":      ("core",    "One row per scheduled arrival. Consecutive rows of a trip define graph edges."),
    "stops.txt":           ("core",    "Graph nodes: stop_id, Hebrew name, latitude/longitude, fare zone."),
    "trips.txt":           ("used",    "Links each trip to its route and service_id (calendar)."),
    "routes.txt":          ("used",    "Route metadata incl. route_type (bus=3, rail=2, light rail=0)."),
    "agency.txt":          ("context", "Operators (Israel Railways, Egged, Dan, ...)."),
    "calendar.txt":        ("context", "Weekly service pattern and validity window per service_id."),
    "translations.txt":    ("unused",  "Multi-language name strings; the project works in Hebrew directly."),
    "fare_rules.txt":      ("unused",  "Fare zone rules - pricing, not topology."),
    "fare_attributes.txt": ("unused",  "Ticket prices - pricing, not topology."),
}

def count_data_rows(path, chunk=8 << 20):
    """Stream the file and count newline bytes; returns rows excluding the header."""
    newlines, last = 0, b"\n"
    with open(path, "rb") as fh:
        while True:
            buf = fh.read(chunk)
            if not buf:
                break
            newlines += buf.count(b"\n")
            last = buf[-1:]
    total_lines = newlines + (0 if last == b"\n" else 1)
    return max(total_lines - 1, 0)

def header_columns(path):
    """Read only the first line. utf-8-sig strips the BOM this feed ships with."""
    with open(path, "r", encoding="utf-8-sig", newline="") as fh:
        return fh.readline().rstrip("\r\n").split(",")

rows = []
missing = []
for name, (tier, purpose) in FILE_PURPOSE.items():
    path = DATA / name
    if not path.exists():
        missing.append(name)
        continue
    size_mb = path.stat().st_size / 1024**2
    do_count = COUNT_ROWS or size_mb < 200
    cols = header_columns(path)
    rows.append({
        "file": name,
        "size_mb": round(size_mb, 2),
        "rows": count_data_rows(path) if do_count else np.nan,
        "columns": len(cols),
        "tier": tier,
        "column_names": ", ".join(cols),
        "purpose": purpose,
    })

if missing:
    raise FileNotFoundError(
        "Missing GTFS file(s): " + ", ".join(missing) + "\n"
        f"Expected inside {DATA}. If stop_times.txt is the missing one, re-run the "
        "gdown cell above; otherwise re-clone the repository."
    )

inventory = pd.DataFrame(rows).sort_values("size_mb", ascending=False).reset_index(drop=True)
inventory.to_csv(STAGE / "tables" / "gtfs_file_inventory.csv",
                 index=False, encoding="utf-8-sig")

print(f"{len(inventory)} GTFS files, "
      f"{inventory['size_mb'].sum():,.1f} MB total, "
      f"{inventory['rows'].sum():,.0f} data rows\n")
inventory[["file", "size_mb", "rows", "columns", "tier", "purpose"]]

## 8. איזו תקופת שירות מכוסה בתצלום מצב זה?

פיד GTFS הוא *תצלום מצב* (snapshot), וכל טענה שנעלה בהמשך על "הרשת" היא למעשה טענה
על לוח הזמנים שהיה בתוקף בחלון זמן מסוים. `calendar.txt` מציין, עבור כל `service_id`,
באילו ימים בשבוע הוא פועל ובין אילו תאריכים הוא תקף (`start_date` / `end_date`,
בפורמט `YYYYMMDD`).

לכן אנו לוקחים את ה-`start_date` המינימלי ואת ה-`end_date` המקסימלי על פני כל
השירותים, כדי לקבל את המעטפת החיצונית של תצלום המצב, וסופרים כמה שירותים פועלים בכל
יום בשבוע. פרופיל ימי השבוע ראוי למבט מסיבה ישראלית מובהקת: שבת היא יום מנוחה ויום
שישי אחר הצהריים הוא יום חלקי, ולכן פיד תקין אמור להראות ירידה נראית לעין בימי שישי
ושבת. אם כך אכן קורה, הנתונים מתנהגים כמצופה.

In [ ]:
calendar = pd.read_csv(DATA / "calendar.txt", dtype=str,
                       keep_default_na=False, encoding="utf-8-sig")

start = pd.to_datetime(calendar["start_date"], format="%Y%m%d")
end = pd.to_datetime(calendar["end_date"], format="%Y%m%d")

feed_start, feed_end = start.min(), end.max()
span_days = (feed_end - feed_start).days + 1

DAYS = ["sunday", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday"]
day_counts = pd.DataFrame({
    "weekday": DAYS,
    "services_running": [(calendar[d] == "1").sum() for d in DAYS],
})
day_counts["share_of_services"] = (day_counts["services_running"] / len(calendar)).round(3)
day_counts.to_csv(STAGE / "tables" / "service_calendar_summary.csv",
                  index=False, encoding="utf-8-sig")

print(f"service_id entries : {len(calendar):,}")
print(f"feed valid from    : {feed_start.date()}")
print(f"feed valid until   : {feed_end.date()}")
print(f"span               : {span_days} days\n")
day_counts

## 9. מבט ראשון בשלוש הטבלאות הבונות את הגרף

מספרים בטבלה נותרים מופשטים עד שרואים את השורות עצמן, ולכן להלן מספר הרשומות
הראשונות של שלושת הקבצים שמהם עשוי הגרף. שתי בחירות טעינה נעשו במכוון ומשמשות
באופן עקבי בכל מחברת בהמשך:

- `dtype=str` - מזהי GTFS הם *מחרוזות אטומות*. לו הרשינו ל-pandas לנחש, `stop_id`
  בערך `"007"` היה הופך למספר השלם `7`, וצירופים מול קובץ ששמר אותו כטקסט היו
  נכשלים בשקט. הכול נשאר טקסט עד שאנו ממירים במפורש.
- `keep_default_na=False` - שדה GTFS ריק פירושו "לא סופק", ולא "מספר חסר". שמירתו
  כ-`""` מונעת דליפת `NaN` לתוך עמודות מחרוזת.

כדאי לשים לב לצורת הנתונים: `stops.txt` מספק קואורדינטות ושם בעברית לכל צומת,
`routes.txt` מספק את אמצעי התחבורה דרך `route_type`, ו-`trips.txt` הוא הגשר
מסוג רבים-לאחד מהרצות בודדות של כלי רכב חזרה אל הקווים.

In [ ]:
def load_gtfs(name, nrows=None):
    """Load a GTFS table as raw strings (see markdown above for why)."""
    path = DATA / name
    if not path.exists():
        raise FileNotFoundError(f"{path} missing - see the setup cells above.")
    return pd.read_csv(path, dtype=str, keep_default_na=False,
                       encoding="utf-8-sig", nrows=nrows)

stops = load_gtfs("stops.txt")
routes = load_gtfs("routes.txt")
agency = load_gtfs("agency.txt")

print("stops.txt  ", stops.shape)
display(stops.head(PREVIEW_ROWS))

print("routes.txt ", routes.shape)
display(routes.head(PREVIEW_ROWS))

trips_head = load_gtfs("trips.txt", nrows=PREVIEW_ROWS)
print("trips.txt  (preview only - full file has ~420k rows)")
display(trips_head)

print("stop_times.txt (preview only - full file has ~15.7M rows)")
display(pd.read_csv(STOP_TIMES, dtype=str, keep_default_na=False,
                    encoding="utf-8-sig", nrows=PREVIEW_ROWS))

## 10. הרכב הפיד: אמצעי תחבורה ומפעילים

שני פילוחים תיאוריים מכוונים את הציפיות לכל מה שיבוא בהמשך.

**לפי `route_type`** (קוד אמצעי התחבורה של GTFS: 0 = חשמלית/רכבת קלה, 2 = רכבת
כבדה, 3 = אוטובוס, וכן הלאה). הדבר חשוב משום שהרשת הארצית היא ברובה המכריע אוטובוסים,
ומכאן שמבנה הגרף המשולב הוא במהותו מבנה רשת האוטובוסים. זו בדיוק הסיבה שהפרויקט מריץ
גם ניתוח נפרד לרכבות בלבד - הרכבת הכבדה היא תת-רשת קטנה, דלילה וכמעט לינארית, שפרופיל
החוסן שלה שונה לחלוטין, ואחרת היא הייתה נבלעת.

**לפי מפעיל.** רשת האוטובוסים בישראל מופעלת בזיכיון על ידי מפעילים אזוריים רבים, ולכן
ספירת קווים לכל מפעיל מראה עד כמה המערכת מפוצלת. זהו גם האיור הראשון עם תוויות
בעברית, ולכן הוא משמש בו-זמנית כמבחן חי ל-patch של ה-bidi מסעיף 5 - אם שמות המפעילים
נקראים נכון מימין לשמאל, מחסנית הציור פועלת במלואה.

In [ ]:
ROUTE_TYPE_LABELS = {
    "0": "tram/light rail", "1": "subway", "2": "rail", "3": "bus",
    "4": "ferry", "5": "cable tram", "6": "aerial lift", "7": "funicular",
    "8": "trolleybus", "715": "demand/other bus",
}

# --- routes per mode ---------------------------------------------------------
route_types = (routes["route_type"].value_counts()
               .rename_axis("route_type").reset_index(name="routes"))
route_types["route_type_label"] = route_types["route_type"].map(ROUTE_TYPE_LABELS).fillna("unknown")
route_types = route_types[["route_type", "route_type_label", "routes"]]
route_types["share"] = (route_types["routes"] / route_types["routes"].sum()).round(4)
route_types.to_csv(STAGE / "tables" / "route_type_distribution.csv",
                   index=False, encoding="utf-8-sig")

fig, ax = plt.subplots(figsize=(7, 4))
labels = route_types["route_type_label"] + " (" + route_types["route_type"] + ")"
ax.bar(labels, route_types["routes"], color="#0f766e")
ax.set_ylabel("routes")
ax.set_title("Routes by GTFS route_type")
ax.tick_params(axis="x", rotation=30)
for lbl in ax.get_xticklabels():
    lbl.set_ha("right")
fig.tight_layout()
fig.savefig(STAGE / "figures" / "route_type_distribution.png", dpi=150)
plt.show()

display(route_types)

# --- routes per operator -----------------------------------------------------
agency_routes = (routes.merge(agency[["agency_id", "agency_name"]], on="agency_id", how="left")
                 .assign(agency_name=lambda d: d["agency_name"].replace("", "unknown"))
                 .groupby("agency_name").size()
                 .sort_values(ascending=False)
                 .rename("routes").reset_index())
agency_routes.to_csv(STAGE / "tables" / "agency_route_counts.csv",
                     index=False, encoding="utf-8-sig")

top = agency_routes.head(TOP_AGENCIES).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top["agency_name"], top["routes"], color="#1d4ed8")
ax.set_xlabel("routes")
ax.set_title(f"Top {TOP_AGENCIES} operators by number of routes")
fig.tight_layout()
fig.savefig(STAGE / "figures" / "routes_per_agency.png", dpi=150)
plt.show()

print(f"{len(agency_routes)} operators in the feed")
display(agency_routes.head(TOP_AGENCIES))

## 11. בדיקת שפיות גיאוגרפית

הבדיקה האחרונה היא הישירה ביותר: לשרטט כל תחנה לפי קו האורך וקו הרוחב שלה, ללא מפה,
ללא היטל וללא עיצוב. אם עמודות הקואורדינטות מפורסרות נכון, קו המתאר של ישראל אמור
פשוט להופיע - הרצועה החופית הצפופה מאשקלון דרך תל אביב ועד חיפה, מסדרון ירושלים
המסתעף פנימה, והנגב הדליל המידלדל דרומה.

זוהי ולידציה שימושית באמת ולא רק תמונה נאה: החלפה בין עמודות lat/lon, בעיה במפריד
העשרוני, או שורות עם קואורדינטות ברירת מחדל `0,0` - כולן היו נראות כאן מיד ובלתי
נראות בטבלה של סטטיסטיקות סיכום. אנו סופרים כמה תחנות בעלות קואורדינטות בלתי שמישות
במקום להשמיטן בשקט.

In [ ]:
coords = stops.copy()
coords["stop_lat"] = pd.to_numeric(coords["stop_lat"], errors="coerce")
coords["stop_lon"] = pd.to_numeric(coords["stop_lon"], errors="coerce")

bad = coords["stop_lat"].isna() | coords["stop_lon"].isna()
out_of_range = (~bad) & ~(coords["stop_lat"].between(29, 34) & coords["stop_lon"].between(33, 36))
print(f"stops total              : {len(coords):,}")
print(f"unparseable coordinates  : {int(bad.sum()):,}")
print(f"outside the Israel bbox  : {int(out_of_range.sum()):,}")

plot_df = coords[~bad & ~out_of_range]
fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(plot_df["stop_lon"], plot_df["stop_lat"], s=0.6, alpha=0.35,
           color="#0f766e", linewidths=0)
ax.set_xlabel("longitude")
ax.set_ylabel("latitude")
ax.set_title(f"All {len(plot_df):,} GTFS stops")
ax.set_aspect(1.15)
fig.tight_layout()
fig.savefig(STAGE / "figures" / "stop_locations.png", dpi=150)
plt.show()

## 12. כתיבת מניפסט השלב

לבסוף אנו שומרים מניפסט JSON קטן המתאר את תצלום המצב של הפיד: מצאי הקבצים, חלון
השירות והספירות המרכזיות. המחברות הבאות טוענות אותו כדי לדווח *מאיזה* תצלום מצב
הגיעו המספרים שלהן, והוא מאפשר לעקוב אחר הסדרה כולה עד לגרסה מתוארכת אחת של הנתונים,
במקום "פיד ה-GTFS" באופן מופשט.

שלב 00 אינו מייצר במכוון שום גרף ושום מדד. החוזה היחיד שלו עם שאר הסדרה הוא: הפיד
נמצא בדיסק, הוא מתפרסר, וזה מה שיש בתוכו.

In [ ]:
manifest = {
    "stage": "00_setup_and_data",
    "data_dir": str(DATA),
    "feed_service_start": str(feed_start.date()),
    "feed_service_end": str(feed_end.date()),
    "feed_span_days": int(span_days),
    "files": int(len(inventory)),
    "total_size_mb": round(float(inventory["size_mb"].sum()), 2),
    "stop_times_rows": (None if pd.isna(inventory.set_index("file").loc["stop_times.txt", "rows"])
                        else int(inventory.set_index("file").loc["stop_times.txt", "rows"])),
    "stops": int(len(stops)),
    "routes": int(len(routes)),
    "agencies": int(len(agency)),
    "service_ids": int(len(calendar)),
    "route_type_counts": dict(zip(route_types["route_type_label"], route_types["routes"].astype(int))),
}

with (STAGE / "feed_manifest.json").open("w", encoding="utf-8") as fh:
    json.dump(manifest, fh, ensure_ascii=False, indent=2)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("\nArtifacts written:")
for p in sorted(STAGE.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(STAGE).as_posix())

## מסקנות

- **הסביבה עובדת.** מחסנית התלויות מיובאת בהצלחה, הקובץ `stop_times.txt` בגודל
  816 MB נמצא בדיסק, כל תשעת קבצי ה-GTFS מתפרסרים, ותוויות בעברית מרונדרות בסדר
  תצוגה. כל כשל שיתרחש בהמשך הוא בעיית ניתוח, לא בעיית התקנה.
- **הפיד הוא תצלום מצב אחד, ולא "הרשת לנצח".** `calendar.txt` נותן חלון תוקף מפורש,
  וכל מסקנה במחברות הבאות מתוחמת אליו. פרופיל ימי השבוע מראה את הירידה הצפויה בימי
  שישי ושבת, וזהו סימן טוב לכך שהנתונים שלמים.
- **הרשת היא ברובה המכריע אוטובוסים.** האוטובוסים שולטים בספירת הקווים בעוצמה כזו
  שהגרף המשולב הוא, מבחינה מבנית, גרף האוטובוסים. הרכבת הכבדה היא שבריר זעיר מהקווים
  - וזו בדיוק הסיבה שהיא מקבלת ניתוח משלה במקום להיקרא מתוך המספרים הארציים, שבהם
  היא בלתי נראית סטטיסטית.
- **רק ארבעה מתוך תשעת הקבצים נושאים טופולוגיה.** `stop_times.txt`, `stops.txt`,
  `trips.txt` ו-`routes.txt` בונים את הגרף; `agency.txt` ו-`calendar.txt` מספקים
  הקשר; שני קבצי התעריפים ו-`translations.txt` אינם רלוונטיים כלל לשאלת קישוריות.
  ראוי לומר במפורש שרשת משוקללת לפי תעריף או לפי זמן נסיעה הייתה מחקר אחר ומעניין
  אף הוא - פרויקט זה אינו מנסה לעשות זאת.
- **מגבלה כנה אחת, שתאמר מראש:** הקשתות שאנו עומדים לבנות מקודדות *טופולוגיית
  שירות*, ולא גיאוגרפיה או זמן נסיעה. קשת פירושה "כלי רכב מתוזמן נוסע ישירות מ-A
  ל-B", משוקללת לפי מספר הנסיעות שעושות זאת. היא אינה אומרת דבר על כמה זמן זה אורך
  או על המרחק. כל תוצאת centrality וחוסן בסדרה זו חייבת להיקרא במונחים אלה.

**הבא בתור:** המחברת `02_graph_construction` קוראת בזרימה את `stop_times.txt`
והופכת את זוגות התחנות העוקבות לגרף התחנות המשוקלל.